In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

In [2]:
import xgboost as xgb
import dill #aneto
import SuperLore
import category_encoders


## Data Loading

In [3]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")




In [4]:
bb


XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0.65, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.025, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=13, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=110, n_jobs=10, num_parallel_tree=None,
              objective='binary:hinge', predictor=None, ...)

In [5]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')



In [6]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [7]:
#carico shapley value full
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

# Data preparation

In [8]:
feature_names = X_test.columns
real_feature_names = X_test.columns

In [9]:
explanations_shap

array([[ 0.09466362,  0.08120622,  0.28542907, ..., -0.07626389,
         0.        , -0.02731891],
       [-0.02177902,  0.07350576, -0.01937944, ...,  0.0781945 ,
         0.        ,  0.96232297],
       [ 0.01969376,  0.08790127, -0.03470209, ..., -0.16089288,
         0.        , -0.26676439],
       ...,
       [ 0.02056669,  0.08696288, -0.03176322, ...,  0.13737915,
         0.        , -0.17642036],
       [-0.00200953,  0.06920489, -0.03739386, ..., -0.07819865,
         0.        , -0.19285882],
       [ 0.04856581, -0.03532057, -0.01439595, ..., -0.01933182,
         0.        ,  0.06414079]])

In [14]:
def data_to_plot(feature_names=feature_names, explanations_shap=explanations_shap, instance=0):
    temp={}
    i=instance
    df_viz = pd.DataFrame(columns = ['type', 'name', 'rname', 'min', 'max', 'q1', 'median', 'q3', 'mean',
           'std', 'feature_importance', 'category', 'count', 'inst', 'op', 'thr',
           'is_continuous', 'thr2'])
    
    bool_f= False
    for j,f in enumerate(feature_names):
        temp[f]= dict()
        temp[f]['feature_importance'] = explanations_shap[0][j]
        for lore_p in objs[i].rule.premises:
            if lore_p.att == f:
                op = lore_p.op
                thr = lore_p.thr
                is_continuous = lore_p.is_continuous
                temp[f]['op']= op
                temp[f]['thr']= thr
                temp[f]['is_continuous']= is_continuous
                bool_f= True
        if len(np.unique(X_train[f]))==2:
            type_f = 'categorical'
            count=np.unique(X_train[f], return_counts= True)
            temp[f]['type']= type_f
            temp[f]['count']= count
        else:
            type_f = 'numeric'
            temp[f]['type']= type_f
            min_f = X_train[f].min()
            temp[f]['min']= min_f
            max_f = X_train[f].max()
            temp[f]['max']= max_f
            q_1 =X_train[f].quantile(0.25)
            temp[f]['q1']= q_1
            median =X_train[f].quantile(0.50)
            temp[f]['median']= median
            q_3=X_train[f].quantile(0.75)
            temp[f]['q3']= q_3
            temp[f]['mean'] = X_train[f].mean()
            temp[f]['std']=X_train[f].std()
            if bool_f == True:
                if (op == '>=' or op == '>'):
                    temp[f]['thr_2'] = max_f
                else:
                    temp[f]['thr_2'] = min_f
            bool_f=False
    df_viz = df_viz.from_dict(temp).T.reset_index().rename(columns={'index':'name'})
    df_viz['rname']=df_viz['name']
    df_viz=df_viz.explode(['count']).explode(['count'])
    df_viz = df_viz[((df_viz['count']!=1) & (df_viz['count']!=0 ))].reset_index(drop=True)
    return df_viz

In [15]:
df=data_to_plot(instance=6)
df

,name,feature_importance,type,min,max,q1,median,q3,mean,std,count,op,thr,is_continuous,thr_2,rname
0,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO,0.094664,numeric,0.0,1.0,0.42743,0.505918,0.559206,0.496342,0.167743,NaN,NaN,NaN,NaN,NaN,PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO
1,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO,0.081206,numeric,0.0,1.0,0.246011,0.366138,0.530745,0.410288,0.223558,NaN,NaN,NaN,NaN,NaN,PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO
2,PRODV_LETTERE_DI_CREDITO_PON,0.285429,numeric,0.0,1.0,0.452953,0.452953,0.452953,0.500201,0.14717,NaN,NaN,NaN,NaN,NaN,PRODV_LETTERE_DI_CREDITO_PON
3,SCADV_FLG_RATA_DIVISA_SEK,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,SCADV_FLG_RATA_DIVISA_SEK
4,OWNER_PRODV_FOREX_PON,0.334717,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3800,NaN,NaN,NaN,NaN,OWNER_PRODV_FOREX_PON
5,OWNER_PRODV_FOREX_PON,0.334717,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,358,NaN,NaN,NaN,NaN,OWNER_PRODV_FOREX_PON
6,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,0.115931,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3975,NaN,NaN,NaN,NaN,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON
7,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON,0.115931,categorical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,183,NaN,NaN,NaN,NaN,OWNER_PRODV_RIMESSE_DOCUMENTARIE_PON
8,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM,0.037198,numeric,0.0,1.0,0.174504,0.333834,0.634134,0.406126,0.285385,NaN,NaN,NaN,NaN,NaN,PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM
9,PN_LC_IMPORT_FLG_ONLY_TY,0.0,numeric,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,PN_LC_IMPORT_FLG_ONLY_TY


# plot function

In [16]:
def plot_rules(dataframe, only_rules=False):   
    def single_rule_plot_qualit(dataframe, rw):
        name= rw['name'].split('=')[0]
        data = dataframe[dataframe['rname'] == name]

        base= alt.Chart(
            data
        ).transform_stack(
            stack='count',
            as_=['count_start','count_end'],
            groupby=['rname'],
            sort=[alt.SortField('count', 'descending')]
        ).transform_calculate(
            midStack='(datum.count_start+datum.count_end)/2'
        )


        bar = base.mark_bar(
            stroke='white',
            color='lightgrey'
        ).encode(
            x='count_start:Q',
            x2='count_end:Q',
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        bar_r = base.mark_bar(
             stroke='#fcc40f'
         ).encode(
            x='count_start:Q',
            x2='count_end:Q',
            color=alt.condition('datum.is_continuous && datum.inst==1',alt.value("#fcc40f"),alt.value('white')),
            opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.0001)),
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        r=base.mark_bar(
            stroke='white'
        ).encode(
            x=alt.X(
                field='count',
                type='quantitative',
                title=None,
            ),
            y=alt.Y(
                field='rname',
                type='nominal',
                axis=None
            ),
            detail='name:N',
            color=alt.condition('datum.is_continuous',alt.value('#fcc40f'),alt.value('white')),
            opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
            tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
        )

        dot =base.mark_point(
            size=70,
            shape='diamond',
            color='black',
            filled=True
        ).encode(
            x=alt.X(
                field='midStack',
                type='quantitative',
                title=None,
            ),
            y=alt.Y(
                field='rname',
                type='nominal',
                axis=None
            ),
            opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
        )


        return alt.layer(bar,bar_r,dot).properties(
            height=20,
            width=300,
        )

    ########
    def single_index_text(dataframe, rw):
        data = dataframe[dataframe['name'] == rw['name']]
        chart = alt.Chart(
            data
        ).transform_calculate(
            label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
        ).mark_text(
            color='black',
            align='left',
            dx=-50,
            fontSize=13
        ).encode(
                text=alt.Text(
                field='label',
                type='nominal',
                title=None
            )
        )
        return chart.properties(
            height=20,
            width=101
        )
    ########
    def single_rule_plot_numeric(dataframe,rw):
        data = dataframe[dataframe['name'] == rw['name']]
        p=alt.Chart(
            data
        ).mark_point(
            color='black' if rw['is_continuous'] == True else 'black',
            size=70,
            shape='diamond',
            filled=True
        ).encode(
            x=alt.X(
                field='inst',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
        )

        t_min = alt.Chart(
            data
        ).mark_text(
            color='black',
            dx=-10,
            align='right'
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None
            ),
            text='min:N'
        )

        t_max = alt.Chart(
            data
        ).mark_text(
            color='black',
            dx=10,
            align='left'
        ).encode(
            x=alt.X(
                field='max',
                type='quantitative',
                title=None
            ),
            text='max:N'
        )



        b =alt.Chart(
            data
        ).mark_bar(
            color='#fcc40f',size=5,
            stroke='white'
        ).encode(
            x=alt.X(
                field='thr',
                type='quantitative',
                title=None,
            ),
            x2='thr2',
            y=alt.Y(
                field='name',
                type='nominal',
                title=None
            ),

        )



        l =alt.Chart(
            data
        ).mark_bar(
            color='grey',size=1
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2='max',
            y=alt.Y(field='name',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
        )


        q1_m = alt.Chart(
            data
        ).mark_bar(
            stroke='white',
            color='lightgrey',
            size=18
        ).encode(
            x=alt.X(
                field='q1',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2 = alt.X2(
                field='median'
            ),
        )

        m_q3 = alt.Chart(
            data
        ).mark_bar(
            stroke='white',
            color='lightgrey',
            size=18,
        ).encode(
            x=alt.X(
                field='median',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
            ),
            x2 = alt.X2(
                field='q3'
            )
        )

        return alt.layer(l,q1_m,m_q3,b,p).properties(
            height=20,
            width=300        
        )
    ########
    def single_feature_importance_plot(dataframe, rw):
        data = dataframe[dataframe['name'] == rw['name']]
        chart = alt.Chart(
            data
        ).mark_bar(
        ).encode(
            x=alt.X(
                field='feature_importance',
                type='quantitative',
                title=None
            ),
            y=alt.Y(
                field='name',
                type='nominal',
                title=None,
                axis=None
            ),
            color=alt.condition('datum.feature_importance > 0', alt.value('#285588'), alt.value('#E36273')),
            tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
        )
        return chart.properties(
            height=20,
            width=100
        )
    ########
    def data_to_plot(
        feature_names=feature_names, real_feature_names=real_feature_names,
        instance_number=None, x_train=None, rules=None,
        feature_importance=None):
        feature_list =[]
        #Convert the list of tuples generated by lime in a dict
        if feature_importance is 'lime':
            lime_dict = {}
            for (key, value) in lime_feature_importance:
                lime_dict.setdefault(key, value)
        for i, el in enumerate(feature_names):
            f ={}
            if el in numeric_columns:
                f['type'] = 'numeric'
                f['name'] = el
                f['rname'] = real_feature_names[i]
                if x_train is not None:
                    f['min'] = x_train[el].min()
                    f['max'] = x_train[el].max()
                    f['q1'] = x_train[el].quantile(0.25)
                    f['median'] = x_train[el].quantile(0.50)
                    f['q3'] = x_train[el].quantile(0.75)
                    f['mean'] = x_train[el].mean()
                    f['std'] = x_train[el].std()
            else:
                f['type'] = 'categorical'
                f['name'] = el
                f['rname'] = el.split('=')[0]
                f['category'] = el.split('=',1)[1]
                if x_train is not None:
                    f['count'] = x_train[el].sum()

            if feature_importance is 'lime':
                f['feature_importance'] = lime_dict[el]
            if feature_importance is 'shap':
                f['feature_importance'] = shap_feature_importance[1][i]
            feature_list.append(f)
        df =pd.DataFrame.from_records(feature_list)

        if instance_number:
            inst = X_train.iloc[instance_number].values
            df['inst'] = inst
        if rules is not None:
            df_rules = pd.DataFrame.from_records(rules)
            df = df.merge(df_rules,how='left',left_on='name',right_on='att')
            df = df.drop('att', axis=1)
            thr2_list=[]
            for i, row in df.iterrows():
                if row['op']== '>' or row['op']== '>=':
                    thr2_list.append(row['max'])
                    continue
                if row['op']== '<' or row['op']== '<=':
                    thr2_list.append(row['min'])
                    continue
                else:
                    thr2_list.append(np.nan)
            df['thr2'] = thr2_list
        df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
        return df
    
    ########
    ti_list=[]
    rp_list=[]
    fi_list=[]
    
    for i, row in dataframe.iterrows():
        if row['inst']!=0:
            if ((only_rules == True) and (row['is_continuous']!= True)):
                pass
            else:
                sti = single_index_text(dataframe, row)
                if row['type']== 'numeric':
                    srp = single_rule_plot_numeric(dataframe, row)
                else:
                    srp = single_rule_plot_qualit(dataframe, row)
                sfi = single_feature_importance_plot(dataframe, row)
                ti_list.append(sti)
                rp_list.append(srp)
                fi_list.append(sfi)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI').resolve_scale(
    x='shared'
)
    final_chart = alt.hconcat(
        fi_concat, rp_concat, ti_concat
    )

    return final_chart.configure(
       # background='#F5F5F5',
        padding=20
    ).configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [17]:
plot_rules(df)

KeyError: 'inst'